This notebook will anonimize the generated data in the simulated data folder. It will get all the images in that folder, give them a random subject ID and resave them in a new folder called "anonimized_simulated_data". It will also generate a csv file with the mapping of the original file names to the new anonimized file names. This will allow us to keep track of the data while also ensuring that the data is anonimized for any future use.

In [15]:
# import necessary libraries
import os
import random
import shutil

# define the path to the simulated data folder and the new folder for anonimized data
simulated_data_folder = "simulated_data"
anonimized_data_folder = "anonimized_simulated_data"

# create the new folder if it doesn't exist
if not os.path.exists(anonimized_data_folder):
    os.makedirs(anonimized_data_folder)


In [16]:
# get all the files in the simulated data folder
files = os.listdir(simulated_data_folder)
# we are going to do this for ERP and full report images separately because the file names have different formats and we want to make sure we are correctly identifying the conditions and subject numbers for each type of image. The new subject IDs will be the same for both types of images to maintain the mapping between the ERP and full report images for each subject.

# create a list to store the mapping of original file names to new anonimized file names
mapping = []

# define the conditions that need to be identified in the file names and the corresponding new file name format
conditions = {
    "reduced": "reduced_sub",
    "reduced delayed": "reduced_delayed",
    "delayed": "delayed_sub",
    "normal": "normal_sub",
    "abolished": "abolished_sub",
}
# filter out only the ones that start with "ERP"
files_ERP = [file for file in files if file.startswith("ERP")]

#filter out the ones that do not start with "ERP"
files_non_ERP = [file for file in files if not file.startswith("ERP")]

# generate a set of random IDs that have already been used to ensure uniqueness
ids = list(range(1000, 10000))
random.shuffle(ids)


# loop through the files and rename them with a random subject ID
for file in files_ERP:
    if file.endswith(".jpg") and "ERP" in file:
        # generate a random subject ID from the list of IDs and remove it from the list to ensure uniqueness
        subject_id = ids.pop()

        # create the new file name
        new_file_name = f"subject_{subject_id}_ERP.jpg"
        # copy the file to the new folder with the new name
        shutil.copy(os.path.join(simulated_data_folder, file), os.path.join(anonimized_data_folder, new_file_name))
        # get condition and subject number from the original file name and add it to the mapping
        for condition in conditions:
            if condition in file:
                condition_name = conditions[condition]
                break
        else:
            condition_name = "unknown_condition"

        # get subject number from the original file name
        subject_number = file.split("_")[-1]
        # get rid of the leading "sub" and the trailing ".jpg"
        subject_number = subject_number.replace("sub", "").replace(".jpg", "")

        # store mapping between this specific subject and condition to the new randomized ID. We will use the same mapping for the full report image of the same subject to maintain the link between the ERP and full report images for each subject. We will store this mapping in a list of dictionaries with keys "condition", "original_subject_number", "original_file_name", and "anonimized_file_name"
        mapping.append({"condition": condition_name, "original_subject_number": subject_number, "original_file_name_erp": file, "anonimized_file_name_erp": new_file_name})



In [17]:

# now we do the same for the full report images, but we use the same subject ID for the corresponding ERP image to maintain the mapping between the ERP and full report images for each subject. We will also add the condition and original subject number to the mapping for the full report images as well.
for file in files_non_ERP:
    if file.endswith(".jpg") and not file.startswith("ERP"):
        # get condition and subject number from the original file name
        for condition in conditions:
            if condition in file:
                condition_name = conditions[condition]
                break
        else:
            condition_name = "unknown_condition"

        # get subject number from the original file name
        subject_number = file.split("_")[-1]
        # get rid of the leading "sub" and the trailing ".jpg"
        subject_number = subject_number.replace("sub", "").replace(".jpg", "")

        # find the corresponding ERP image for this full report image in the mapping list to get the new anonimized file name with the same subject ID
        for mapping_entry in mapping:
            if mapping_entry["original_subject_number"] == subject_number and mapping_entry["condition"] == condition_name:
                new_file_name = mapping_entry["anonimized_file_name_erp"].replace("ERP", "full_report")
                break
        else:
            new_file_name = f"subject_{random.randint(1000, 9999)}_full_report.jpg"

        # copy the file to the new folder with the new name
        shutil.copy(os.path.join(simulated_data_folder, file), os.path.join(anonimized_data_folder, new_file_name))
        # add the mapping of the full report to the mapping list with the same condition and original subject number as the corresponding ERP image. We will add a new key "original_file_name_full_report" to the mapping dictionary to store the original file name of the full report image and we will use the same "anonimized_file_name" key to store the new anonimized file name for the full report image as well. This way we can maintain the link between the ERP and full report images for each subject while also keeping track of the original file names for both types of images in the mapping.
        for mapping_entry in mapping:
            if mapping_entry["original_subject_number"] == subject_number and mapping_entry["condition"] == condition_name:
                mapping_entry["original_file_name_full_report"] = file
                mapping_entry["anonimized_file_name_full_report"] = new_file_name
                break
        else:
            mapping.append({"condition": condition_name, "original_subject_number": subject_number, "original_file_name_full_report": file, "anonimized_file_name_full_report": new_file_name})


In [18]:
# save the mapping to a csv file
import pandas as pd
mapping_df = pd.DataFrame(mapping)
mapping_df.to_csv(os.path.join(anonimized_data_folder, "mapping.csv"), index=False)
